In [36]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import svm
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, confusion_matrix, recall_score
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

In [37]:
df = pd.read_csv('DB_LIMPIA.csv', sep=";")
df.head(5)

,Unnamed: 0,VIC_SEXO,VIC_EDAD,TOTAL_HIJOS,VIC_ESCOLARIDAD,VIC_EST_CIV,VIC_GRUPET,VIC_TRABAJA,VIC_DISC,VIC_REL_AGR,...,HEC_TIPAGRE,INST_DONDE_DENUNCIO,AGR_SEXO,AGR_EDAD,AGR_ESCOLARIDAD,AGR_EST_CIV,AGR_GRUPET,AGR_TRABAJA,INST_DENUN_HECHO,MEDIDAS_SEGURIDAD
0,0,Mujeres,11,-1,Primaria,Desconocido,Ladino,No,Desconocido,Hijos(as),...,Física-sexual,NaN,Hombres,58,Primaria,Soltero,Ladino,No,Ministerio Público,Desconocido
1,1,Hombres,4,-1,Desconocido,Desconocido,Ladino,Desconocido,No,Hijos(as),...,Física-psicológica,NaN,Mujeres,27,Primaria,Soltero,Ladino,Si,Procuraduría de los Derechos Humanos,Desconocido
2,2,Hombres,11,-1,Primaria,Desconocido,Ladino,No,No,Hijos(as),...,Física-psicológica,NaN,Hombres,35,Ninguna,Unido,Ladino,Si,Procuraduría de los Derechos Humanos,Desconocido
3,3,Mujeres,6,-1,Desconocido,Desconocido,Ladino,Desconocido,No,Hijos(as),...,Psicológica,NaN,Hombres,35,Primaria,Soltero,Ladino,Si,Organismo Judicial,Si
4,4,Hombres,11,-1,Primaria,Desconocido,Ladino,No,No,Hijos(as),...,Psicológica,NaN,Hombres,35,Primaria,Soltero,Ladino,Si,Organismo Judicial,Si


In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 358998 entries, 0 to 358997
Data columns (total 25 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   Unnamed: 0           358998 non-null  int64
 1   VIC_SEXO             358998 non-null  str  
 2   VIC_EDAD             358998 non-null  int64
 3   TOTAL_HIJOS          358998 non-null  int64
 4   VIC_ESCOLARIDAD      358998 non-null  str  
 5   VIC_EST_CIV          358998 non-null  str  
 6   VIC_GRUPET           358998 non-null  str  
 7   VIC_TRABAJA          358998 non-null  str  
 8   VIC_DISC             358998 non-null  str  
 9   VIC_REL_AGR          358998 non-null  str  
 10  OTRAS_VICTIMAS       358998 non-null  int64
 11  HEC_DIA              358998 non-null  int64
 12  HEC_MES              358998 non-null  str  
 13  HEC_ANO              358998 non-null  int64
 14  HEC_AREA             358998 non-null  str  
 15  HEC_TIPAGRE          358998 non-null  str  
 16  INST_DONDE_DE

Se procede a crear un conjunto de datos en el que se encuentren etiquetados en VICTIMAS y AGRESORES, como se ha hecho en el primer proyecto.

In [39]:
# Se crea el DataFrame de víctimas.
df_victimas = df.loc[:, ['VIC_SEXO', 'VIC_EDAD', 'VIC_ESCOLARIDAD', 'VIC_EST_CIV', 'VIC_GRUPET', 'VIC_TRABAJA']]
df_victimas.columns = ['SEXO', 'EDAD', 'ESCOLARIDAD', 'ESTADO_CIVIL', 'GRUPO_ETNICO', 'TRABAJA']
df_victimas['ETIQUETA'] = 'VICTIMA'

# Se crea el DataFrame de agresores.
df_agresores = df.loc[:, ['AGR_SEXO', 'AGR_EDAD', 'AGR_ESCOLARIDAD', 'AGR_EST_CIV', 'AGR_GRUPET', 'AGR_TRABAJA']]
df_agresores.columns = ['SEXO', 'EDAD', 'ESCOLARIDAD', 'ESTADO_CIVIL', 'GRUPO_ETNICO', 'TRABAJA']
df_agresores['ETIQUETA'] = 'AGRESOR'

# Se crea el DataFrame de ambos conjuntos etiquetados.
DF = pd.concat([df_victimas, df_agresores])
DF.head(10)

,SEXO,EDAD,ESCOLARIDAD,ESTADO_CIVIL,GRUPO_ETNICO,TRABAJA,ETIQUETA
0,Mujeres,11,Primaria,Desconocido,Ladino,No,VICTIMA
1,Hombres,4,Desconocido,Desconocido,Ladino,Desconocido,VICTIMA
2,Hombres,11,Primaria,Desconocido,Ladino,No,VICTIMA
3,Mujeres,6,Desconocido,Desconocido,Ladino,Desconocido,VICTIMA
4,Hombres,11,Primaria,Desconocido,Ladino,No,VICTIMA
5,Hombres,1,Desconocido,Desconocido,Ladino,Desconocido,VICTIMA
6,Mujeres,10,Ninguna,Desconocido,Ladino,No,VICTIMA
7,Hombres,8,Ninguna,Desconocido,Desconocido,Desconocido,VICTIMA
8,Mujeres,10,Primaria,Desconocido,Ladino,Desconocido,VICTIMA
9,Mujeres,10,Primaria,Desconocido,Ladino,Si,VICTIMA


Se presentan ahora las estadísticas del objetivo:

In [40]:
DF['ETIQUETA'].describe()

count      717996
unique          2
top       VICTIMA
freq       358998
Name: ETIQUETA, dtype: object

Tratamiento de N.A.

In [41]:
df = DF.fillna(DF.median(numeric_only=True))#Tratamiento de NA

In [42]:
#Se procesan las etiquetas, al retirarlas del dataframe
categoria=[]
for x in df["ETIQUETA"]:
    categoria.append(x)
df = df.drop(columns=['ETIQUETA'])

In [71]:
len(categoria)

717996

Dado que hay varias variables tipo string, se procederá ha realizar una codificación tanto entera como binaria, según sea el caso, de estas variables

In [53]:
df

,SEXO,EDAD,ESCOLARIDAD,ESTADO_CIVIL,GRUPO_ETNICO,TRABAJA
0,Mujeres,11,Primaria,Desconocido,Ladino,No
1,Hombres,4,Desconocido,Desconocido,Ladino,Desconocido
2,Hombres,11,Primaria,Desconocido,Ladino,No
3,Mujeres,6,Desconocido,Desconocido,Ladino,Desconocido
4,Hombres,11,Primaria,Desconocido,Ladino,No
...,...,...,...,...,...,...
358993,Hombres,25,Básico,Desconocido,Ladino,No
358994,Hombres,30,Básico,Desconocido,Ladino,Si
358995,Hombres,28,Básico,Casado,Ladino,No
358996,Hombres,35,Diversificado,Desconocido,Ladino,Si


In [64]:
#Se codifica en enteros
df['ESCOLARIDAD']=df['ESCOLARIDAD'].astype('category').cat.codes
df['SEXO']=df['SEXO'].astype('category').cat.codes
df['ESTADO_CIVIL']=df['ESTADO_CIVIL'].astype('category').cat.codes
df['GRUPO_ETNICO']=df['GRUPO_ETNICO'].astype('category').cat.codes
df['TRABAJA']=df['TRABAJA'].astype('category').cat.codes
df

,SEXO,EDAD,ESCOLARIDAD,ESTADO_CIVIL,GRUPO_ETNICO,TRABAJA
0,1,11,4,1,2,1
1,0,4,1,1,2,0
2,0,11,4,1,2,1
3,1,6,1,1,2,0
4,0,11,4,1,2,1
...,...,...,...,...,...,...
358993,0,25,0,1,2,1
358994,0,30,0,1,2,2
358995,0,28,0,0,2,1
358996,0,35,2,1,2,2


SEPARACIÓN DE DATOS

In [78]:
X_train, X_test, y_train, y_test = train_test_split(df, categoria, test_size=0.3, random_state=42, stratify=categoria)

In [79]:
print(len(X_train))
print(len(y_train))

502597
502597


Se escalan los datos

In [80]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [81]:
print(len(X_train))
print(len(y_train))

502597
502597


Se procede con el modelo

Modelo SVM con kernel lineal

In [ ]:
param_dist = {
    'kernel': ['linear'],
    'C': [0.1], 
    'gamma': [0.1, ]
}

modelo1.fit(X_train, y_train)

y_pred1 = modelo1.predict(X_test)

Modelo SVM con kernel rbf

Modelo SVM optimizado